In [1]:
import pandas as pd 

In [2]:
from pathlib import Path

project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "data" / "raw").is_dir()
)
data_dir = project_root / "data" / "raw"

customers_df = pd.read_csv(data_dir / "olist_customers_dataset.csv")
geolocations_df = pd.read_csv(data_dir / "olist_geolocation_dataset.csv")
orders_items_df = pd.read_csv(data_dir / "olist_order_items_dataset.csv")
orders_payments_df = pd.read_csv(data_dir / "olist_order_payments_dataset.csv")
orders_reviews_df = pd.read_csv(data_dir / "olist_order_reviews_dataset.csv")
orders_df = pd.read_csv(data_dir / "olist_orders_dataset.csv")
products_df = pd.read_csv(data_dir / "olist_products_dataset.csv")
sellers_df = pd.read_csv(data_dir / "olist_sellers_dataset.csv")
categories_df = pd.read_csv(data_dir / "product_category_name_translation.csv")

In [3]:
datasets = {"customer": customers_df, 
            "geolocation": geolocations_df, 
            "order_items": orders_items_df, 
            "order_payments": orders_payments_df, 
            "order_reviews": orders_reviews_df, 
            "orders": orders_df, 
            "products": products_df, 
            "sellers": sellers_df, 
            "categories": categories_df}

In [4]:
#Make dataframe of table of Dataset,Rows,Columns
data = []
for name, df in datasets.items():
    data.append([name, df.shape[0], df.shape[1]])

df_table = pd.DataFrame(data, columns=["Dataset", "Rows", "Columns"])
print(df_table)

          Dataset     Rows  Columns
0        customer    99441        5
1     geolocation  1000163        5
2     order_items   112650        7
3  order_payments   103886        5
4   order_reviews    99224        7
5          orders    99441        8
6        products    32951        9
7         sellers     3095        4
8      categories       71        2


In [6]:
for name, df in datasets.items():
    print(f"\n{'=' * 50}")
    print(f"{name.upper()} DATASET")
    print(f"{'=' * 50}")
    df.info()


CUSTOMER DATASET


<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB

GEOLOCATION DATASET
<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  str    
 4  

In [7]:
#Create a summary table for every dataset that includes:
#Dataset	Column	Missing Count	Missing %
summary_data = []
for name, df in datasets.items():
    for column in df.columns:
        missing_count = df[column].isnull().sum()
        missing_percentage = (missing_count / len(df)) * 100
        if missing_count > 0:
            summary_data.append([name, column, missing_count, missing_percentage])

summary_df = pd.DataFrame(
    summary_data,
    columns=["Dataset", "Column", "Missing Count", "Missing %"]
)
summary_df

,Dataset,Column,Missing Count,Missing %
0,order_reviews,review_comment_title,87656,88.341530
1,order_reviews,review_comment_message,58247,58.702532
2,orders,order_approved_at,160,0.160899
3,orders,order_delivered_carrier_date,1783,1.793023
4,orders,order_delivered_customer_date,2965,2.981668
5,products,product_category_name,610,1.851234
6,products,product_name_lenght,610,1.851234
7,products,product_description_lenght,610,1.851234
8,products,product_photos_qty,610,1.851234
9,products,product_weight_g,2,0.006070


In [8]:
# Check for exact duplicate rows
duplicate_data = []
for name, df in datasets.items():
    duplicate_count = df.duplicated().sum()
    duplicate_data.append([name, len(df), duplicate_count])

duplicate_summary_df = pd.DataFrame(
    duplicate_data,
    columns=["Dataset", "Total Rows", "Duplicate Rows"]
)
duplicate_summary_df

,Dataset,Total Rows,Duplicate Rows
0,customer,99441,0
1,geolocation,1000163,261831
2,order_items,112650,0
3,order_payments,103886,0
4,order_reviews,99224,0
5,orders,99441,0
6,products,32951,0
7,sellers,3095,0
8,categories,71,0


In [9]:
# Create a primary-key summary for every dataset
primary_key_definitions = [
    ("Customers", "customer", ["customer_id"]),
    ("Orders", "orders", ["order_id"]),
    ("Products", "products", ["product_id"]),
    ("Sellers", "sellers", ["seller_id"]),
    ("Categories", "categories", ["product_category_name_english"]),
    ("Order Items", "order_items", ["order_id", "order_item_id"]),
    ("Payments", "order_payments", ["order_id"]),
    ("Reviews", "order_reviews", ["review_id"]),
    ("Geolocation", "geolocation", ["geolocation_zip_code_prefix"]),
]

primary_key_data = []
for display_name, dataset_name, key_columns in primary_key_definitions:
    df = datasets[dataset_name]
    duplicate_key_count = df.duplicated(subset=key_columns).sum()
    null_key_count = df[key_columns].isnull().any(axis=1).sum()
    is_valid_pk = duplicate_key_count == 0 and null_key_count == 0

    primary_key_data.append([
        display_name,
        " + ".join(key_columns),
        duplicate_key_count,
        null_key_count,
        is_valid_pk,
    ])

primary_key_summary_df = pd.DataFrame(
    primary_key_data,
    columns=["Dataset", "PK", "Duplicate Keys", "Null Keys", "Valid"]
)
primary_key_summary_df

,Dataset,PK,Duplicate Keys,Null Keys,Valid
0,Customers,customer_id,0,0,True
1,Orders,order_id,0,0,True
2,Products,product_id,0,0,True
3,Sellers,seller_id,0,0,True
4,Categories,product_category_name_english,0,0,True
5,Order Items,order_id + order_item_id,0,0,True
6,Payments,order_id,4446,0,False
7,Reviews,review_id,814,0,False
8,Geolocation,geolocation_zip_code_prefix,981148,0,False


1.Customers, Orders, Products, Sellers, Categories, and Order Items have valid primary keys (with Order Items using a composite key).
2.order_id is not a valid primary key for the Payments table because an order can have multiple payment records.
3.review_id is not unique, indicating it cannot be treated as a reliable primary key without further investigation.
4.The Geolocation table does not have a natural primary key based on geolocation_zip_code_prefix, suggesting it should be treated as a reference dataset rather than a transactional entity.

In [10]:
# ============================================================================
# Foreign Key Relationship Definitions
# ============================================================================
# Define the expected foreign key relationships between the Olist datasets.
# Each tuple contains:
# (Child Table Name, Child Dataset Key, Foreign Key Column,
#  Parent Table Name, Parent Dataset Key, Parent Primary Key Column)
foreign_key_definitions = [
    ("Orders", "orders", "customer_id", "Customers", "customer", "customer_id"),
    ("Order Items", "order_items", "order_id", "Orders", "orders", "order_id"),
    ("Order Items", "order_items", "product_id", "Products", "products", "product_id"),
    ("Order Items", "order_items", "seller_id", "Sellers", "sellers", "seller_id"),
    ("Payments", "order_payments", "order_id", "Orders", "orders", "order_id"),
    ("Reviews", "order_reviews", "order_id", "Orders", "orders", "order_id"),
    ("Geolocation", "geolocation", "geolocation_zip_code_prefix", None, None, None),
]

foreign_key_data = []
for child_name, child_key, foreign_key, parent_name, parent_key, primary_key in foreign_key_definitions:
    if parent_key is None:
        continue

    child_df = datasets[child_key]
    parent_df = datasets[parent_key]
    non_null_foreign_keys = child_df[foreign_key].dropna()
    parent_values = parent_df[primary_key].dropna()
    missing_foreign_keys = non_null_foreign_keys[
        ~non_null_foreign_keys.isin(parent_values)
    ]
    null_foreign_keys = child_df[foreign_key].isnull().sum()
    is_valid_fk = missing_foreign_keys.empty

    foreign_key_data.append([
        child_name,
        foreign_key,
        parent_name,
        primary_key,
        missing_foreign_keys.nunique(),
        null_foreign_keys,
        is_valid_fk,
    ])

foreign_key_summary_df = pd.DataFrame(
    foreign_key_data,
    columns=[
        "Child Dataset",
        "Foreign Key",
        "Parent Dataset",
        "Parent Key",
        "Missing References",
        "Null Foreign Keys",
        "Valid FK",
    ]
)
foreign_key_summary_df

,Child Dataset,Foreign Key,Parent Dataset,Parent Key,Missing References,Null Foreign Keys,Valid FK
0,Orders,customer_id,Customers,customer_id,0,0,True
1,Order Items,order_id,Orders,order_id,0,0,True
2,Order Items,product_id,Products,product_id,0,0,True
3,Order Items,seller_id,Sellers,seller_id,0,0,True
4,Payments,order_id,Orders,order_id,0,0,True
5,Reviews,order_id,Orders,order_id,0,0,True


In [11]:
entity_id_columns = {
    "Orders": "order_id",
    "Products": "product_id",
    "Sellers": "seller_id",
}

entity_item_summary = []
for entity, id_column in entity_id_columns.items():
    items_per_entity = orders_items_df.groupby(id_column).size()
    entity_item_summary.append({
        "Entity": entity,
        "Unique IDs": items_per_entity.size,
        "Min Items": items_per_entity.min(),
        "Max Items": items_per_entity.max(),
        "Avg Items": items_per_entity.mean(),
        "IDs with >1 Item": (items_per_entity > 1).sum(),
    })

entity_item_summary_df = pd.DataFrame(entity_item_summary)
entity_item_summary_df

,Entity,Unique IDs,Min Items,Max Items,Avg Items,IDs with >1 Item
0,Orders,98666,1,21,1.141731,9803
1,Products,32951,1,527,3.418713,14834
2,Sellers,3095,1,2033,36.397415,2586


1.Orders → Order Items

Each order can contain multiple order-item records. The maximum observed is 21 items per order, confirming a 1:M relationship.

2.Products → Order Items

Products can appear across multiple order-item records. The maximum observed is 527 order-item records for a single product, confirming a 1:M relationship.

3.Sellers → Order Items

Sellers can be associated with multiple order-item records. The maximum observed is 2,033 records for a single seller, confirming a 1:M relationship.

4.And one overall observation:

The order_items table represents the line-item level of the transaction, where each row corresponds to an item within an order and is associated with a product and seller.

5.The likely future grain will be something like:

One row = one product/seller line item within an order.

In [12]:
customer_id_counts = customers_df.groupby("customer_unique_id")["customer_id"].nunique()

customer_identity_summary = pd.DataFrame({
    "Metric": [
        "Unique customer_id",
        "Unique customer_unique_id",
        "customer_unique_id values with >1 customer_id",
    ],
    "Count": [
        customers_df["customer_id"].nunique(),
        customers_df["customer_unique_id"].nunique(),
        (customer_id_counts > 1).sum(),
    ],
})

customer_identity_summary

,Metric,Count
0,Unique customer_id,99441
1,Unique customer_unique_id,96096
2,customer_unique_id values with >1 customer_id,2997


In [15]:
orders_per_customer_id = orders_df.groupby("customer_id").size()

summary = pd.DataFrame({
    "Metric": [
        "Unique customer_id represented in orders",
        "Minimum orders per customer_id",
        "Maximum orders per customer_id",
        "Average orders per customer_id",
        "Number of customer_ids with >1 order",
    ],
    "Value": [
        orders_per_customer_id.index.nunique(),
        orders_per_customer_id.min(),
        orders_per_customer_id.max(),
        orders_per_customer_id.mean(),
        (orders_per_customer_id > 1).sum(),
    ]
})

summary

,Metric,Value
0,Unique customer_id represented in orders,99441.0
1,Minimum orders per customer_id,1.0
2,Maximum orders per customer_id,1.0
3,Average orders per customer_id,1.0
4,Number of customer_ids with >1 order,0.0


In [16]:
orders_customer_map = orders_df.merge(
    customers_df[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left",
)

orders_per_real_customer = (
    orders_customer_map.groupby("customer_unique_id")
    .size()
)

summary = pd.DataFrame({
    "Metric": [
        "Unique customer_unique_id represented in orders",
        "Minimum orders per real customer",
        "Maximum orders per real customer",
        "Average orders per real customer",
        "Number of real customers with >1 order",
    ],
    "Value": [
        orders_per_real_customer.index.nunique(),
        orders_per_real_customer.min(),
        orders_per_real_customer.max(),
        orders_per_real_customer.mean(),
        (orders_per_real_customer > 1).sum(),
    ],
})

summary

,Metric,Value
0,Unique customer_unique_id represented in orders,96096.000000
1,Minimum orders per real customer,1.000000
2,Maximum orders per real customer,17.000000
3,Average orders per real customer,1.034809
4,Number of real customers with >1 order,2997.000000


Under Relationship & Cardinality Observations, I'd write:

Customer → Orders: At the customer_id level, each customer record is associated with exactly one order. However, customer_unique_id represents the underlying customer identity, and one real customer can be associated with multiple customer IDs/orders. Therefore, the business-level relationship between customer_unique_id and orders is 1:M. The maximum observed orders for a real customer is 17.

In [19]:
payments_per_order = orders_payments_df.groupby("order_id").size()
reviews_per_order = orders_reviews_df.groupby("order_id").size()

summary_data = []
for name, series in [
    ("payments_per_order", payments_per_order),
    ("reviews_per_order", reviews_per_order),
]:
    summary_data.append({
        "Dataset": name,
        "Unique orders represented": series.index.nunique(),
        "Minimum records per order": series.min(),
        "Maximum records per order": series.max(),
        "Average records per order": series.mean(),
        "Number of orders with >1 record": (series > 1).sum(),
    })

order_record_summary = pd.DataFrame(summary_data)
order_record_summary


,Dataset,Unique orders represented,Minimum records per order,Maximum records per order,Average records per order,Number of orders with >1 record
0,payments_per_order,99440,1,29,1.044710,2961
1,reviews_per_order,98673,1,3,1.005584,547


In [20]:
categories_per_product = (
    products_df
    .groupby("product_id")["product_category_name"]
    .nunique()
)

missing_category_count = products_df["product_category_name"].isna().sum()

product_category_summary = pd.DataFrame({
    "Metric": [
        "Unique products",
        "Minimum categories per product",
        "Maximum categories per product",
        "Average categories per product",
        "Products with more than one category",
        "Number of products with a missing category",
    ],
    "Value": [
        categories_per_product.index.nunique(),
        categories_per_product.min(),
        categories_per_product.max(),
        categories_per_product.mean(),
        (categories_per_product > 1).sum(),
        missing_category_count,
    ],
})

product_category_summary


,Metric,Value
0,Unique products,32951.000000
1,Minimum categories per product,0.000000
2,Maximum categories per product,1.000000
3,Average categories per product,0.981488
4,Products with more than one category,0.000000
5,Number of products with a missing category,610.000000


In [32]:
product_categories = products_df["product_category_name"].dropna()
lookup_categories = categories_df["product_category_name"].dropna()

missing_from_lookup = product_categories[
    ~product_categories.isin(lookup_categories)
]

product_category_lookup_check = pd.DataFrame({
    "Metric": [
        "Number of non-null product category values",
        "Number of unique product category values",
        "Number of unique category values in lookup",
        "Number of product rows with unmatched category",
        "Number of unique product category values that do not exist in lookup"
    ],
    "Value": [
        product_categories.size,
        product_categories.nunique(),
        lookup_categories.nunique(),
        missing_from_lookup.size,
        missing_from_lookup.unique(),
    ],
})

product_category_lookup_check

,Metric,Value
0,Number of non-null product category values,32341
1,Number of unique product category values,73
2,Number of unique category values in lookup,71
3,Number of product rows with unmatched category,13
4,Number of unique product category values that ...,"[pc_gamer, portateis_cozinha_e_preparadores_de..."


In [33]:
missing_category_counts = (
    missing_from_lookup
    .value_counts()
    .reset_index()
)

missing_category_counts.columns = [
    "Unmatched Category",
    "Product Count"
]

missing_category_counts

,Unmatched Category,Product Count
0,portateis_cozinha_e_preparadores_de_alimentos,10
1,pc_gamer,3


In [36]:
geolocations_df["geolocation_zip_code_prefix"].nunique()
geolocations_df["geolocation_zip_code_prefix"].value_counts().head()

geolocation_zip_code_prefix
24220    1146
24230    1102
38400     965
35500     907
11680     879
Name: count, dtype: int64

In [38]:
geolocations_df.duplicated(
    subset=[
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng",
        "geolocation_city",
        "geolocation_state"
    ]
).sum()

np.int64(261831)

In [39]:
geolocations_df.duplicated(
    subset=[
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng"
    ]
).sum()

np.int64(280009)

In [40]:
unique_geolocation_points = geolocations_df[
    [
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng",
        "geolocation_city",
        "geolocation_state"
    ]
].drop_duplicates()

unique_geolocation_points.shape

(738332, 5)

In [45]:
locations_per_zip = (
    unique_geolocation_points
    .groupby("geolocation_zip_code_prefix")
    .size()
)

geolocation_zip_summary = pd.DataFrame({
    "Metric": [
        "Unique ZIP prefixes",
        "Minimum locations per ZIP",
        "Maximum locations per ZIP",
        "Average locations per ZIP",
        "ZIP prefixes with >1 location",
    ],
    "Value": [
        locations_per_zip.size,
        locations_per_zip.min(),
        locations_per_zip.max(),
        locations_per_zip.mean(),
        (locations_per_zip > 1).sum(),
    ],
})

geolocation_zip_summary

,Metric,Value
0,Unique ZIP prefixes,19015.000000
1,Minimum locations per ZIP,1.000000
2,Maximum locations per ZIP,779.000000
3,Average locations per ZIP,38.828925
4,ZIP prefixes with >1 location,17823.000000


Better description

For the raw source, we can currently say:

The geolocation table contains repeated geographical observations associated with ZIP-code prefixes. After removing exact duplicates across all five columns, there are 738,332 unique geographical records covering 19,015 ZIP prefixes. A single ZIP prefix can correspond to many geographical records

In [44]:
cities_per_zip = (
    unique_geolocation_points
    .groupby("geolocation_zip_code_prefix")["geolocation_city"]
    .nunique()
)

geolocation_city_summary = pd.DataFrame({
    "Metric": [
        "Minimum cities per ZIP",
        "Maximum cities per ZIP",
        "Average cities per ZIP",
        "ZIPs with >1 city",
    ],
    "Value": [
        cities_per_zip.min(),
        cities_per_zip.max(),
        cities_per_zip.mean(),
        (cities_per_zip > 1).sum(),
    ],
})

geolocation_city_summary

,Metric,Value
0,Minimum cities per ZIP,1.000000
1,Maximum cities per ZIP,5.000000
2,Average cities per ZIP,1.467631
3,ZIPs with >1 city,8556.000000


In [47]:
states_per_zip = (
    unique_geolocation_points
    .groupby("geolocation_zip_code_prefix")["geolocation_state"]
    .nunique()
)

geolocation_state_summary = pd.DataFrame({
    "Metric": [
        "Minimum states per ZIP",
        "Maximum states per ZIP",
        "Average states per ZIP",
        "ZIPs with >1 state",
    ],
    "Value": [
        states_per_zip.min(),
        states_per_zip.max(),
        states_per_zip.mean(),
        (states_per_zip > 1).sum(),
    ],
})

geolocation_state_summary

,Metric,Value
0,Minimum states per ZIP,1.000000
1,Maximum states per ZIP,2.000000
2,Average states per ZIP,1.000421
3,ZIPs with >1 state,8.000000


In [48]:
multi_state_zips = states_per_zip[states_per_zip > 1]

multi_state_zips

geolocations_df[
    geolocations_df["geolocation_zip_code_prefix"].isin(
        multi_state_zips.index
    )
].sort_values("geolocation_zip_code_prefix")

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
21728,2116,-23.522700,-46.587546,sao paulo,SP
21759,2116,-23.518459,-46.584128,sao paulo,SP
21788,2116,-23.518459,-46.584128,sao paulo,SP
22060,2116,-23.519065,-46.584610,sao paulo,SP
22193,2116,-23.519739,-46.585151,são paulo,SP
...,...,...,...,...,...
847871,80630,-25.468930,-49.280779,curitiba,PR
847872,80630,-25.469776,-49.273230,curitiba,PR
847875,80630,-25.466519,-49.268213,curitiba,PR
847878,80630,-25.461689,-49.272160,curitiba,PR


In [50]:
coordinates_per_zip = (
    unique_geolocation_points
    .groupby("geolocation_zip_code_prefix")
    .apply(
        lambda x: x[["geolocation_lat", "geolocation_lng"]]
        .drop_duplicates()
        .shape[0]
    )
)

geolocation_coordinate_summary = pd.DataFrame({
    "Metric": [
        "Minimum coordinates per ZIP",
        "Maximum coordinates per ZIP",
        "Average coordinates per ZIP",
        "ZIPs with >1 coordinate",
    ],
    "Value": [
        coordinates_per_zip.min(),
        coordinates_per_zip.max(),
        coordinates_per_zip.mean(),
        (coordinates_per_zip > 1).sum(),
    ],
})

geolocation_coordinate_summary

,Metric,Value
0,Minimum coordinates per ZIP,1.000000
1,Maximum coordinates per ZIP,746.000000
2,Average coordinates per ZIP,37.872942
3,ZIPs with >1 coordinate,17781.000000


One row represents a geolocation observation containing a ZIP-code prefix, latitude, longitude, city, and state. Multiple observations can exist for the same ZIP prefix, and the raw dataset also contains exact duplicate observations.

In [51]:
customer_zips = customers_df["customer_zip_code_prefix"].dropna()

geolocation_zips = geolocations_df[
    "geolocation_zip_code_prefix"
].dropna()

customer_zip_check = pd.DataFrame({
    "Metric": [
        "Unique customer ZIP prefixes",
        "Unique geolocation ZIP prefixes",
        "Customer ZIP prefixes not found in geolocation",
    ],
    "Value": [
        customer_zips.nunique(),
        geolocation_zips.nunique(),
        customer_zips[
            ~customer_zips.isin(geolocation_zips)
        ].nunique(),
    ],
})

customer_zip_check

,Metric,Value
0,Unique customer ZIP prefixes,14994
1,Unique geolocation ZIP prefixes,19015
2,Customer ZIP prefixes not found in geolocation,157


In [53]:
unmatched_customer_zips = customer_zips[
    ~customer_zips.isin(geolocation_zips)
]

affected_customer_records = customers_df[
    customers_df["customer_zip_code_prefix"].isin(
        unmatched_customer_zips
    )
].shape[0]

customer_zip_unmatched_summary = pd.DataFrame({
    "Metric": [
        "Unique unmatched customer ZIP prefixes",
        "Customer records with unmatched ZIP",
    ],
    "Value": [
        unmatched_customer_zips.nunique(),
        affected_customer_records,
    ],
})

customer_zip_unmatched_summary

,Metric,Value
0,Unique unmatched customer ZIP prefixes,157
1,Customer records with unmatched ZIP,278


In [54]:
seller_zips = sellers_df["seller_zip_code_prefix"].dropna()
geolocation_zips = geolocations_df["geolocation_zip_code_prefix"].dropna()

unmatched_seller_zips = seller_zips[
    ~seller_zips.isin(geolocation_zips)
]

affected_seller_records = sellers_df[
    sellers_df["seller_zip_code_prefix"].isin(
        unmatched_seller_zips
    )
].shape[0]

seller_zip_check = pd.DataFrame({
    "Metric": [
        "Unique seller ZIP prefixes",
        "Unique geolocation ZIP prefixes",
        "Unique unmatched seller ZIP prefixes",
        "Seller records with unmatched ZIP",
        "Percentage of seller records affected",
    ],
    "Value": [
        seller_zips.nunique(),
        geolocation_zips.nunique(),
        unmatched_seller_zips.nunique(),
        affected_seller_records,
        (affected_seller_records / len(sellers_df)) * 100,
    ],
})

seller_zip_check

,Metric,Value
0,Unique seller ZIP prefixes,2246.000000
1,Unique geolocation ZIP prefixes,19015.000000
2,Unique unmatched seller ZIP prefixes,7.000000
3,Seller records with unmatched ZIP,7.000000
4,Percentage of seller records affected,0.226171


Our conclusion

Customers:

157 unique customer ZIP prefixes are not present in the geolocation dataset, affecting 278 customer records (~0.28%).

Sellers:

7 unique seller ZIP prefixes are not present in the geolocation dataset, affecting 7 seller records (~0.23%).

This tells us that geolocation coverage is generally high, despite some unmatched ZIP prefixes.

Geolocation profiling conclusions
-1,000,163 raw records across 5 columns.
-All columns are non-null.
-19,015 unique ZIP prefixes.
-geolocation_zip_code_prefix is not a primary key.
-261,831 exact duplicate rows exist.
-After removing exact duplicates, 738,332 unique five-column records remain.
-A ZIP prefix can have multiple geographical observations.
-Maximum unique coordinate observations for a ZIP = 746.
-17,781 ZIP prefixes have more than one coordinate.
-A ZIP can map to multiple cities; 8,556 ZIPs have >1 city.
-Almost every ZIP maps to one state, but 8 ZIPs map to 2 states.
-Customer → Geolocation coverage is high, with 278 customers (~0.28%) affected by unmatched ZIP prefixes.
-Seller → Geolocation coverage is high, with 7 sellers (~0.23%) affected.
-ZIP prefix should not be treated as a unique geolocation identifier.

PK
Must uniquely identify a row
FK
References a key in another table
Can repeat in the child table
Lookup column
Used to match records
Does NOT necessarily uniquely identify a row

In [56]:
customers_df.merge(
    geolocations_df,
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix"
)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,14409,-20.509897,-47.397866,franca,SP
1,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,14409,-20.497396,-47.399241,franca,SP
2,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,14409,-20.510459,-47.399553,franca,SP
3,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,14409,-20.480940,-47.394161,franca,SP
4,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,14409,-20.515413,-47.398194,franca,SP
...,...,...,...,...,...,...,...,...,...,...
15083450,274fa6071e5e17fe303b9748641082c8,84732c5050c01db9b23e19ba39899398,6703,cotia,SP,6703,-23.599369,-46.905603,cotia,SP
15083451,274fa6071e5e17fe303b9748641082c8,84732c5050c01db9b23e19ba39899398,6703,cotia,SP,6703,-23.593577,-46.910112,cotia,SP
15083452,274fa6071e5e17fe303b9748641082c8,84732c5050c01db9b23e19ba39899398,6703,cotia,SP,6703,-23.584425,-46.892014,cotia,SP
15083453,274fa6071e5e17fe303b9748641082c8,84732c5050c01db9b23e19ba39899398,6703,cotia,SP,6703,-23.595022,-46.918546,cotia,SP


In [57]:
candidate_key_map = {
    "Customers": ["customer_id"],
    "Orders": ["order_id"],
    "Products": ["product_id"],
    "Sellers": ["seller_id"],
    "Categories": ["product_category_name_english"],
    "Order Items": ["order_id", "order_item_id"],
    "Payments": ["order_id"],
    "Reviews": ["review_id"],
    "Geolocation": ["geolocation_zip_code_prefix"],
}

datasets = {
    "Customers": customers_df,
    "Orders": orders_df,
    "Products": products_df,
    "Sellers": sellers_df,
    "Categories": categories_df,
    "Order Items": orders_items_df,
    "Payments": orders_payments_df,
    "Reviews": orders_reviews_df,
    "Geolocation": geolocations_df,
}

rows = []
for dataset_name, df in datasets.items():
    key_cols = candidate_key_map[dataset_name]
    duplicate_count = df.duplicated(subset=key_cols).sum()
    null_count = df[key_cols].isnull().any(axis=1).sum()
    if len(key_cols) == 1:
        unique_key_count = df[key_cols[0]].nunique(dropna=False)
    else:
        unique_key_count = df[key_cols].drop_duplicates().shape[0]

    rows.append({
        "Dataset": dataset_name,
        "Rows": len(df),
        "Unique IDs / candidate key": unique_key_count,
        "Duplicate count": duplicate_count,
        "Null count": null_count,
    })

pd.DataFrame(rows)


,Dataset,Rows,Unique IDs / candidate key,Duplicate count,Null count
0,Customers,99441,99441,0,0
1,Orders,99441,99441,0,0
2,Products,32951,32951,0,0
3,Sellers,3095,3095,0,0
4,Categories,71,71,0,0
5,Order Items,112650,112650,0,0
6,Payments,103886,99440,4446,0
7,Reviews,99224,98410,814,0
8,Geolocation,1000163,19015,981148,0


![image.png](attachment:image.png)

In [58]:
payment_dups = orders_payments_df.duplicated(subset=["order_id", "payment_sequential"]).sum()
review_dups = orders_reviews_df.duplicated(subset=["review_id", "order_id"]).sum()

print(payment_dups)
print(review_dups)

0
0


In [61]:
orders_df["order_purchase_timestamp"] = pd.to_datetime(orders_df["order_purchase_timestamp"])
orders_df["order_approved_at"] = pd.to_datetime(orders_df["order_approved_at"])
orders_df["order_delivered_carrier_date"] = pd.to_datetime(orders_df["order_delivered_carrier_date"])
orders_df["order_delivered_customer_date"] = pd.to_datetime(orders_df["order_delivered_customer_date"])

orders_df["purchase_to_approval"] = (
    orders_df["order_approved_at"]
    - orders_df["order_purchase_timestamp"]
)

orders_df["approval_to_carrier"] = (
    orders_df["order_delivered_carrier_date"]
    - orders_df["order_approved_at"]
)

orders_df["carrier_to_customer"] = (
    orders_df["order_delivered_customer_date"]
    - orders_df["order_delivered_carrier_date"]
)

orders_df["purchase_to_delivery"] = (
    orders_df["order_delivered_customer_date"]
    - orders_df["order_purchase_timestamp"]
)

In [62]:
lifecycle_summary = pd.DataFrame({
    "Metric": [
        "Purchase → Approval negative",
        "Approval → Carrier negative",
        "Carrier → Customer Delivery negative",
        "Purchase → Customer Delivery negative",
    ],
    "Count": [
        (orders_df["purchase_to_approval"] < pd.Timedelta(0)).sum(),
        (orders_df["approval_to_carrier"] < pd.Timedelta(0)).sum(),
        (orders_df["carrier_to_customer"] < pd.Timedelta(0)).sum(),
        (orders_df["purchase_to_delivery"] < pd.Timedelta(0)).sum(),
    ]
})

lifecycle_summary

,Metric,Count
0,Purchase → Approval negative,0
1,Approval → Carrier negative,1359
2,Carrier → Customer Delivery negative,23
3,Purchase → Customer Delivery negative,0


Data Quality Observation: 1,359 orders have a carrier handover timestamp earlier than the approval timestamp, and 23 orders have a customer delivery timestamp earlier than the carrier handover timestamp. These records require further investigation before any corrective transformation is applied.

In [63]:
status_delivery_summary = pd.DataFrame({
    "Metric": [
        "Delivered orders",
        "Delivered orders without customer delivery date",
        "Non-delivered orders with customer delivery date",
    ],
    "Count": [
        (orders_df["order_status"] == "delivered").sum(),

        (
            (orders_df["order_status"] == "delivered") &
            (orders_df["order_delivered_customer_date"].isna())
        ).sum(),

        (
            (orders_df["order_status"] != "delivered") &
            (orders_df["order_delivered_customer_date"].notna())
        ).sum(),
    ]
})

status_delivery_summary

,Metric,Count
0,Delivered orders,96478
1,Delivered orders without customer delivery date,8
2,Non-delivered orders with customer delivery date,6


Data Quality Observation: 8 orders are marked as delivered but have no customer delivery timestamp, while 6 orders with a non-delivered status contain a customer delivery timestamp. These records should be investigated before applying transformation rules.

In [65]:
numeric_validation = pd.DataFrame({
    "Metric": [
        "Invalid review scores",
        "Negative item prices",
        "Negative freight values",
        "Negative payment values",
        "Invalid payment installments",
        "Non-positive product weights",
        "Non-positive product length",
        "Non-positive product height",
        "Non-positive product width",
    ],
    "Count": [
        (~orders_reviews_df["review_score"].between(1, 5)).sum(),
        (orders_items_df["price"] < 0).sum(),
        (orders_items_df["freight_value"] < 0).sum(),
        (orders_payments_df["payment_value"] < 0).sum(),
        (orders_payments_df["payment_installments"] <= 0).sum(),
        (products_df["product_weight_g"] <= 0).sum(),
        (products_df["product_length_cm"] <= 0).sum(),
        (products_df["product_height_cm"] <= 0).sum(),
        (products_df["product_width_cm"] <= 0).sum(),
    ]
})

numeric_validation

,Metric,Count
0,Invalid review scores,0
1,Negative item prices,0
2,Negative freight values,0
3,Negative payment values,0
4,Invalid payment installments,2
5,Non-positive product weights,4
6,Non-positive product length,0
7,Non-positive product height,0
8,Non-positive product width,0


In [66]:
final_profiling_summary = pd.DataFrame({
    "Area": [
        "Missing Values",
        "Duplicate Records",
        "Primary Keys",
        "Foreign Keys",
        "Order Lifecycle",
        "Order Status",
        "Numeric / Business Rules",
        "Geolocation",
    ],
    "Status": [
        "Issues identified",
        "Issues identified in Geolocation",
        "Validated",
        "Validated",
        "Exceptions identified",
        "Exceptions identified",
        "Exceptions identified",
        "Issues identified",
    ],
    "Key Findings": [
        "Missing comments, timestamps, and product attributes",
        "261,831 duplicate geolocation rows",
        "Valid PKs identified for core transactional/dimension tables",
        "All tested FK references are valid",
        "1,359 approval/carrier and 23 carrier/delivery inconsistencies",
        "8 delivered orders missing delivery date; 6 non-delivered with delivery date",
        "2 invalid installment records; 4 non-positive product weights",
        "ZIP prefixes map to multiple geographic observations",
    ]
})

final_profiling_summary

,Area,Status,Key Findings
0,Missing Values,Issues identified,"Missing comments, timestamps, and product attr..."
1,Duplicate Records,Issues identified in Geolocation,"261,831 duplicate geolocation rows"
2,Primary Keys,Validated,Valid PKs identified for core transactional/di...
3,Foreign Keys,Validated,All tested FK references are valid
4,Order Lifecycle,Exceptions identified,"1,359 approval/carrier and 23 carrier/delivery..."
5,Order Status,Exceptions identified,8 delivered orders missing delivery date; 6 no...
6,Numeric / Business Rules,Exceptions identified,2 invalid installment records; 4 non-positive ...
7,Geolocation,Issues identified,ZIP prefixes map to multiple geographic observ...
